# Collapsing Calo Energy Deposits from Duplicated Particles

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx
import matplotlib.cm as cm
import h5py
import awkward as ak

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch

## Roadmap

1. Load a full-truth calo event
2. Load a reduced-truth calo event
3. Implement a particle aggregation function
4. Re-write the calo hit and calo contribution collections after aggregation
5. Prototype a re-writing of the edm4hep file, with the collapsed calo deposits

## Minimal Loading

In [36]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root"
event = EDM4hepEvent(edm_input_file, event_index=0, condense_calo=False)

Augmenting particles
Ready with tracker hits Index(['cellID', 'time', 'x', 'y', 'z', 'particle_id', 'detector', 'r', 'R',
       'phi', 'theta', 'eta'],
      dtype='object')
Augmenting particle hit counts with tracker hits


In [37]:
print(f"""
Number of calo hits: {len(event.get_calo_hits_df())}
Number of calo contributions: {len(event.get_calo_contributions_df())}
Number of particles: {len(event.get_particles_df())}
Number of tracker hits: {len(event.get_tracker_hits_df())}
""")


Number of calo hits: 1243953
Number of calo contributions: 6038426
Number of particles: 880327
Number of tracker hits: 240622



In [38]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_mini_pilot/ttbar/v7/runs/0/edm4hep.root"
event2 = EDM4hepEvent(edm_input_file, event_index=0)

Calo condensation took 2.2774 seconds
Augmenting particles
Ready with tracker hits Index(['cellID', 'time', 'x', 'y', 'z', 'particle_id', 'detector', 'r', 'R',
       'phi', 'theta', 'eta'],
      dtype='object')
Augmenting particle hit counts with tracker hits


In [39]:
print(f"""
Number of calo hits: {len(event2.get_calo_hits_df())}
Number of calo contributions: {len(event2.get_calo_contributions_df())}
Number of particles: {len(event2.get_particles_df())}
Number of tracker hits: {len(event2.get_tracker_hits_df())}
""")


Number of calo hits: 1120404
Number of calo contributions: 1611754
Number of particles: 190319
Number of tracker hits: 217781



In [40]:
tracker_hits_df = event.get_tracker_hits_df()

In [41]:
calo_hits_df = event.get_calo_hits_df()

In [42]:
calo_contributions_df = event.get_calo_contributions_df()

In [43]:
print("Memory usage:")
print(f"tracker_hits_df: {tracker_hits_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"calo_hits_df: {calo_hits_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"calo_contributions_df: {calo_contributions_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


Memory usage:
tracker_hits_df: 36.22 MB
calo_hits_df: 162.53 MB
calo_contributions_df: 673.77 MB


In [44]:
tracker_hits_df2 = event2.get_tracker_hits_df()

In [45]:
calo_hits_df2 = event2.get_calo_hits_df()

In [46]:
calo_contributions_df2 = event2.get_calo_contributions_df()

In [47]:
print("Condensed memory usage:")
print(f"tracker_hits_df: {tracker_hits_df2.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"calo_hits_df: {calo_hits_df2.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"calo_contributions_df: {calo_contributions_df2.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


Condensed memory usage:
tracker_hits_df: 32.77 MB
calo_hits_df: 154.93 MB
calo_contributions_df: 179.84 MB


In [58]:
calo_hits_df

,cellID,energy,x,y,z,contribution_begin,contribution_end,detector,r,R,phi,theta,eta
0,104145539527933968,5.172027e-04,719.657898,-1058.644897,1881.900024,0,14,ECalBarrelCollection,1280.092407,2276.001709,-0.973762,0.597322,1.178078
1,50102494323496976,1.691220e-06,545.476929,1156.974487,902.700012,14,15,ECalBarrelCollection,1279.114990,1565.567749,1.130233,0.956240,0.657348
2,18278703426723737616,1.027635e-07,1459.449951,-102.000000,-3049.800049,15,16,ECalBarrelCollection,1463.010010,3382.554932,-0.069776,2.694311,-1.480845
3,18577387126204432,5.202330e-05,-1162.829468,-531.341553,336.600006,16,17,ECalBarrelCollection,1278.474243,1322.042236,-2.712984,1.313356,0.260332
4,18577387126237200,5.268137e-05,-1167.495117,-533.274109,336.600006,17,18,ECalBarrelCollection,1283.520996,1326.923462,-2.713125,1.314324,0.259331
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243948,12947956309262612,2.090424e-04,466.364319,1499.801392,3749.500000,6038419,6038421,HCalEndcapCollection,1570.636841,4065.175293,1.269324,0.396686,1.604522
1243949,12103535674065172,3.217657e-04,513.346008,1417.383423,3698.500000,6038421,6038422,HCalEndcapCollection,1507.481323,3993.920654,1.223313,0.387035,1.629795
1243950,9570265178603796,2.114068e-05,595.443970,1158.424194,3647.500000,6038422,6038423,HCalEndcapCollection,1302.497681,3873.081055,1.096002,0.342980,1.753360
1243951,10696160790479124,1.477808e-08,542.609558,1270.265625,3647.500000,6038423,6038424,HCalEndcapCollection,1381.303711,3900.289307,1.167096,0.362009,1.698227


In [59]:
calo_contributions_df

,energy,time,particle_id,hit_index,cellID,x,y,z,detector
0,6.181428e-05,367.544739,84496,0,104145539527933968,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
1,9.740763e-05,367.545288,84496,0,104145539527933968,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
2,7.868026e-06,367.551208,84496,0,104145539527933968,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
3,7.590641e-06,367.551300,84496,0,104145539527933968,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
4,5.001049e-06,367.551422,84496,0,104145539527933968,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
...,...,...,...,...,...,...,...,...,...
6038421,3.217657e-04,52.863934,879566,211086,12103535674065172,513.346008,1417.383423,3698.500000,HCalEndcapCollection
6038422,2.114068e-05,229.844788,879422,211087,9570265178603796,595.443970,1158.424194,3647.500000,HCalEndcapCollection
6038423,1.477808e-08,5697.791504,879422,211088,10696160790479124,542.609558,1270.265625,3647.500000,HCalEndcapCollection
6038424,2.523652e-09,5751.806152,879422,211089,10414685813768468,548.462280,1240.842163,3647.500000,HCalEndcapCollection


In [60]:
calo_contributions_df['cellID'].isin(calo_hits_df['cellID']).sum()

np.int64(6038426)

In [8]:
# We are going to aggregate duplicated contributions from each particle to each cell
calo_contributions_df["energy_time"] = calo_contributions_df["energy"] * calo_contributions_df["time"]
aggregated_calo_contributions_df = calo_contributions_df.groupby(["cellID", "particle_id"]).agg({"energy": "sum", "time": "mean", "energy_time": "sum"}).reset_index()
aggregated_calo_contributions_df["time"] = aggregated_calo_contributions_df["energy_time"] / aggregated_calo_contributions_df["energy"]
aggregated_calo_contributions_df.drop(columns=["energy_time"], inplace=True)

In [9]:
aggregated_calo_contributions_df

,cellID,particle_id,energy,time
0,6328339,644397,1.122357e-05,549.441895
1,6332435,24126,6.498919e-06,114.604050
2,6338579,60152,1.460476e-04,6.312902
3,6342675,73327,1.773009e-04,85.054756
4,6363155,754,2.722708e-06,467.353851
...,...,...,...,...
2048453,18446744069421002771,29742,6.076018e-07,415.364563
2048454,18446744069423329296,682710,1.613427e-04,1.637509
2048455,18446744069423591440,682597,2.324135e-04,1.766586
2048456,18446744069423892496,11771,1.476583e-04,2.756681


In [10]:
# reset the index, and merge back to get x, y, z, and detector
aggregated_calo_contributions_df = aggregated_calo_contributions_df.merge(calo_hits_df[["cellID", "x", "y", "z", "detector"]], on="cellID", how="left", validate="many_to_one")

In [11]:
cat = pd.Categorical(
    aggregated_calo_contributions_df['cellID'],
    categories=calo_hits_df['cellID'],
    ordered=True,
)

new_calo_contributions_df = (
    aggregated_calo_contributions_df
      .assign(cellID=cat)
      .sort_values(['cellID', 'particle_id'], kind='mergesort')
      .reset_index(drop=True)
)

In [12]:
new_calo_contributions_df

,cellID,particle_id,energy,time,x,y,z,detector
0,104145539527933968,84496,5.172027e-04,367.550293,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
1,50102494323496976,84495,1.691220e-06,1210.152344,545.476929,1156.974487,902.700012,ECalBarrelCollection
2,18278703426723737616,84493,1.027635e-07,2059.835205,1459.449951,-102.000000,-3049.800049,ECalBarrelCollection
3,18577387126204432,84493,5.202330e-05,37.163723,-1162.829468,-531.341553,336.600006,ECalBarrelCollection
4,18577387126237200,84493,5.268137e-05,37.891846,-1167.495117,-533.274109,336.600006,ECalBarrelCollection
...,...,...,...,...,...,...,...,...
2048453,12947956309262612,879566,2.090424e-04,89.865150,466.364319,1499.801392,3749.500000,HCalEndcapCollection
2048454,12103535674065172,879566,3.217657e-04,52.863934,513.346008,1417.383423,3698.500000,HCalEndcapCollection
2048455,9570265178603796,879422,2.114068e-05,229.844803,595.443970,1158.424194,3647.500000,HCalEndcapCollection
2048456,10696160790479124,879422,1.477808e-08,5697.791504,542.609558,1270.265625,3647.500000,HCalEndcapCollection


In [13]:
calo_hits_df

,cellID,energy,x,y,z,contribution_begin,contribution_end,detector,r,R,phi,theta,eta
0,104145539527933968,5.172027e-04,719.657898,-1058.644897,1881.900024,0,14,ECalBarrelCollection,1280.092407,2276.001709,-0.973762,0.597322,1.178078
1,50102494323496976,1.691220e-06,545.476929,1156.974487,902.700012,14,15,ECalBarrelCollection,1279.114990,1565.567749,1.130233,0.956240,0.657348
2,18278703426723737616,1.027635e-07,1459.449951,-102.000000,-3049.800049,15,16,ECalBarrelCollection,1463.010010,3382.554932,-0.069776,2.694311,-1.480845
3,18577387126204432,5.202330e-05,-1162.829468,-531.341553,336.600006,16,17,ECalBarrelCollection,1278.474243,1322.042236,-2.712984,1.313356,0.260332
4,18577387126237200,5.268137e-05,-1167.495117,-533.274109,336.600006,17,18,ECalBarrelCollection,1283.520996,1326.923462,-2.713125,1.314324,0.259331
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243948,12947956309262612,2.090424e-04,466.364319,1499.801392,3749.500000,6038419,6038421,HCalEndcapCollection,1570.636841,4065.175293,1.269324,0.396686,1.604522
1243949,12103535674065172,3.217657e-04,513.346008,1417.383423,3698.500000,6038421,6038422,HCalEndcapCollection,1507.481323,3993.920654,1.223313,0.387035,1.629795
1243950,9570265178603796,2.114068e-05,595.443970,1158.424194,3647.500000,6038422,6038423,HCalEndcapCollection,1302.497681,3873.081055,1.096002,0.342980,1.753360
1243951,10696160790479124,1.477808e-08,542.609558,1270.265625,3647.500000,6038423,6038424,HCalEndcapCollection,1381.303711,3900.289307,1.167096,0.362009,1.698227


In [14]:
counts_per_cell = new_calo_contributions_df.groupby('cellID', sort=False).size()

counts = (
    calo_hits_df['cellID']
      .map(counts_per_cell)
      .fillna(0)
      .astype(np.int64)
)

/tmp/ipykernel_1779067/1088270573.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts_per_cell = new_calo_contributions_df.groupby('cellID', sort=False).size()


In [15]:
counts

0          1
1          1
2          1
3          1
4          1
          ..
1243948    1
1243949    1
1243950    1
1243951    1
1243952    1
Name: cellID, Length: 1243953, dtype: int64

In [16]:
cum = counts.cumsum()
new_end = cum.to_numpy()
new_begin = (cum - counts).to_numpy()

In [18]:
new_end

array([      1,       2,       3, ..., 2048456, 2048457, 2048458],
      shape=(1243953,))

In [19]:
new_begin

array([      0,       1,       2, ..., 2048455, 2048456, 2048457],
      shape=(1243953,), dtype=int64)

In [20]:
(new_begin == new_end).sum()

np.int64(0)

In [21]:
(calo_hits_df.contribution_begin == calo_hits_df.contribution_end).sum()

np.int64(0)

In [22]:
calo_hits_df['contribution_begin'] = new_begin
calo_hits_df['contribution_end'] = new_end

In [23]:
calo_hits

NameError: name 'calo_hits' is not defined

In [27]:
assert len(new_calo_contributions_df) == int(new_end[-1])

# Energy conservation per cell
orig_sums = calo_contributions_df.groupby('cellID')['energy'].sum()
new_sums = new_calo_contributions_df.groupby('cellID')['energy'].sum()
diff = (orig_sums - new_sums).abs()

print('Max energy diff per cell:', diff.max())
print('Num cells with diff > 1e-8:', (diff > 1e-8).sum())

/tmp/ipykernel_1779067/2401124250.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  new_sums = new_calo_contributions_df.groupby('cellID')['energy'].sum()


Max energy diff per cell: 7.450581e-09
Num cells with diff > 1e-8: 0
